# 02 · 🔵 No paramétricas y transformaciones (Python)

**Opcional.** Qué hacer cuando los supuestos del ANOVA fallan: **transformar** (Box-Cox) o usar **Kruskal-Wallis**.

> Teoría: [`../teoria/05-no-parametricas-transformaciones.md`](../teoria/05-no-parametricas-transformaciones.md).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(7)

## 1. Transformación de Box-Cox

Generamos una respuesta con **varianza creciente con la media** (heterocedástica).

In [ ]:
medias = [10, 20, 40, 80]
filas = []
for i, m in enumerate(medias):
    y = m * np.exp(np.random.normal(0, 0.30, size=8))  # ruido multiplicativo
    filas += [(f'T{i+1}', v) for v in y]
dfh = pd.DataFrame(filas, columns=['trat', 'y'])
dfh.groupby('trat')['y'].agg(['mean', 'std'])

La desviación estándar crece con la media (patrón multiplicativo). Box-Cox elige la transformación óptima.

In [ ]:
y = dfh['y'].values
y_bc, lam = stats.boxcox(y)
print(f'lambda óptimo = {lam:.3f}  (~0 -> log; ~0.5 -> raíz; ~1 -> ninguna)')

$\lambda\approx 0$ confirma que el **logaritmo** estabiliza la varianza.

In [ ]:
dfh['log_y'] = np.log(dfh['y'])
print('SD por grupo (original):')
print(dfh.groupby('trat')['y'].std().round(2))
print('\nSD por grupo (log):')
print(dfh.groupby('trat')['log_y'].std().round(3))

En escala logarítmica las desviaciones se igualan: la heterocedasticidad desaparece y el ANOVA puede aplicarse sobre `log_y`.

## 2. Prueba de Kruskal-Wallis

Alternativa no paramétrica al ANOVA: trabaja con **rangos**, sin asumir normalidad.

In [ ]:
coag = pd.read_csv('../datos/tiempo-coagulacion.csv')
grupos = [g['tiempo'].values for _, g in coag.groupby('dieta')]
H, p = stats.kruskal(*grupos)
print(f'H = {H:.3f},  p = {p:.5f}')

Con $p\approx0.0007$ se **rechaza** la igualdad de distribuciones, coherente con el ANOVA paramétrico del notebook 01. El post-hoc no paramétrico es la prueba de **Dunn** (`scikit_posthocs.posthoc_dunn`).

## 3. ¿Cuándo cada cosa?

1. Heterocedasticidad/asimetría → **Box-Cox**, reanalizar.
2. Sin transformación útil u respuesta ordinal → **Kruskal-Wallis** (+ Dunn).
3. Solo varianzas desiguales → **ANOVA de Welch**.

> Si los datos sí son normales, Kruskal-Wallis pierde algo de potencia frente al ANOVA.